# E-commerce App Review Analytics: Sentiment Classification and Customer Experience Improvement

## Purpose and objectives

This MBA capstone examines the available reviews for Amazon Shopping, Flipkart, and Myntra to describe observed ratings and text, assess a text classifier against the dataset's existing sentiment labels, and surface keyword-based candidate customer issues. The purpose is to inform careful customer-experience investigation, not to claim causal app effects or validated customer sentiment.

The project objectives are to:

1. Describe app, rating, existing sentiment-label, and review-length distributions.
2. Compare a majority-class baseline with a text-only classifier on a stratified holdout.
3. Summarize keyword-based candidate issue matches overall and by app.
4. Translate the observed patterns into cautious business considerations and identify validation needs.

## Research questions

- What rating, existing-label, and review-length patterns are present in the available cleaned dataset?
- How well does a TF-IDF and Logistic Regression model agree with the existing labels compared with a majority-class baseline, and which classes remain difficult to predict?
- Which reviews match configured issue-related keywords, and how do candidate-match counts vary across the three apps?
- What customer-experience questions merit further validation, given the limits of the data and methods?

**Evidence convention:** Tables and metrics identified as documented results below come from the existing project reports. Results produced by the lightweight notebook cells are newly computed from the local cleaned CSV when those cells are run. The notebook is initially saved with all code cells unexecuted.


## 2. Environment and imports

The existing environment check recorded Python 3.14.7 and Pandas 3.0.6. This section uses only the standard library and Pandas for the notebook's read-only summaries. It does not install packages or invoke the project pipelines.

**Rerun prerequisites:** Start the notebook from the repository root and provide an authorized cleaned dataset at `data/processed/google_play_reviews_cleaned.csv`. Use a compatible Python environment with Pandas, IPython, and SciPy available. The project does not specify dependency versions or installation commands.


In [2]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

print(f'Python: {sys.version.split()[0]}')
print(f'Pandas: {pd.__version__}')


Python: 3.14.7
Pandas: 3.0.6


## 3. Dataset loading

Load the existing cleaned file using its repository-relative path. The cell reads the CSV only; it does not change or export any data. The cleaned dataset is local project data and is not included in a fresh public clone.


In [3]:
CLEANED_DATA_PATH = Path('data') / 'processed' / 'google_play_reviews_cleaned.csv'
if not CLEANED_DATA_PATH.is_file():
    raise FileNotFoundError(
        f'Cleaned dataset not found at {CLEANED_DATA_PATH.resolve()}. '
        'Run this notebook from the project root and confirm the local data file is available.'
    )

reviews = pd.read_csv(CLEANED_DATA_PATH, encoding='utf-8-sig')
print(f'Loaded {len(reviews):,} rows and {len(reviews.columns)} columns from {CLEANED_DATA_PATH}.')


Loaded 5,840 rows and 15 columns from data\processed\google_play_reviews_cleaned.csv.


## 4. Dataset overview and data-quality checks

The validation report documents 5,840 records and 14 source columns; the cleaned file contains an additional `clean_text_new` field. The checks below are read-only and display column-level missingness, exact duplicate-row count, blank review-text counts, and the observed rating range. Missingness here means Pandas null values; blank strings are counted separately. These outputs are newly computed if the cell is run.


In [4]:
required_columns = {
    'app_name', 'review_text', 'rating', 'sentiment', 'clean_text_new'
}
missing_required_columns = required_columns.difference(reviews.columns)
if missing_required_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_required_columns)}')

print(f'Dataset shape: {reviews.shape[0]:,} rows x {reviews.shape[1]} columns')
print(f'Exact duplicate rows: {reviews.duplicated().sum():,}')
print(f'Blank review_text values: {reviews["review_text"].fillna("").str.strip().eq("").sum():,}')
print(f'Blank clean_text_new values: {reviews["clean_text_new"].fillna("").str.strip().eq("").sum():,}')

missing_by_column = reviews.isna().sum().rename('missing_cells').to_frame()
display(missing_by_column)
print('Observed rating values:', sorted(reviews['rating'].dropna().unique().tolist()))


Dataset shape: 5,840 rows x 15 columns
Exact duplicate rows: 0
Blank review_text values: 0
Blank clean_text_new values: 0


,missing_cells
app_name,0
package_id,0
review_id,0
review_text,0
rating,0
thumbs_up_count,0
review_date,0
review_version,763
app_version,0
collected_at_utc,0


Observed rating values: [1, 2, 3, 4, 5]


## 5. Text preprocessing methodology

The existing implementation in `src/review_analytics/cleaning.py` reads `review_text` to create `clean_text_new`, while retaining every source column and row. Its rules are deliberately conservative:

1. Normalize Unicode to NFC (canonical composition).
2. Convert text to lowercase.
3. Standardize whitespace by splitting on whitespace and joining with one ordinary space, trimming leading and trailing whitespace.

Missing or whitespace-only input becomes an empty string. The implementation does not deliberately remove URLs, punctuation, numbers, negation, or other tokens, and it does not change `rating` or `sentiment`. The cleaning report documents 5,840 input and output rows, no excluded rows, and 2,379 rows where `clean_text_new` differs from `review_text`. Those are previously documented checks; this notebook does not rerun the cleaning function or write a cleaned file.


## 6. Exploratory data analysis (EDA)

The existing EDA report describes app counts, app-level rating and label mixes, and character and whitespace-token lengths computed from `clean_text_new`. It reports 5,840 reviews across Amazon Shopping (1,964), Flipkart (1,900), and Myntra (1,976); mean review length is 82.77 characters and 15.05 whitespace-separated tokens. The 500-character maximum is observed, but the available evidence does not establish that it reflects truncation.

The following lightweight Pandas summaries are notebook computations, not a rerun of `src/review_analytics/eda.py`. They do not create figures or files and do not display review text.


In [5]:
length_summary_input = pd.DataFrame({
    'app_name': reviews['app_name'],
    'characters': reviews['clean_text_new'].fillna('').astype(str).str.len(),
    'whitespace_tokens': reviews['clean_text_new'].fillna('').astype(str).str.split().str.len(),
})

app_record_counts = reviews['app_name'].value_counts(dropna=False).rename_axis('app_name').to_frame('records')
length_summary = length_summary_input.groupby('app_name').agg(
    records=('characters', 'count'),
    mean_characters=('characters', 'mean'),
    median_characters=('characters', 'median'),
    max_characters=('characters', 'max'),
    mean_whitespace_tokens=('whitespace_tokens', 'mean'),
    median_whitespace_tokens=('whitespace_tokens', 'median'),
    max_whitespace_tokens=('whitespace_tokens', 'max'),
).round(2)

display(app_record_counts)
display(length_summary)


,records
app_name,
Myntra,1976
Amazon Shopping,1964
Flipkart,1900


,records,mean_characters,median_characters,max_characters,mean_whitespace_tokens,median_whitespace_tokens,max_whitespace_tokens
app_name,,,,,,,
Amazon Shopping,1964,155.63,105.0,500,28.58,19.0,106
Flipkart,1900,32.08,8.0,500,5.86,2.0,102
Myntra,1976,59.09,19.0,500,10.45,3.0,100


## 7. Rating and sentiment-label distribution

The distributions below describe the values present in the dataset. **Sentiment-label provenance is unverified.** The project audit found exact alignment with the tested mapping (ratings 1-2 negative, rating 3 neutral, ratings 4-5 positive), but does not establish who or what assigned the labels. They must not be represented as independently human-annotated or as validated sentiment ground truth. The report's distributions are documented findings; the following tables are recomputed by this notebook if run.


In [6]:
rating_counts = reviews['rating'].value_counts(dropna=False).sort_index().rename_axis('rating').to_frame('records')
label_counts = reviews['sentiment'].value_counts(dropna=False).rename_axis('existing_label').to_frame('records')
rating_label_counts = pd.crosstab(reviews['rating'], reviews['sentiment'], dropna=False)

display(rating_counts)
display(label_counts)
display(rating_label_counts)


,records
rating,
1,1562
2,249
3,279
4,438
5,3312


,records
existing_label,
positive,3750
negative,1811
neutral,279


sentiment,negative,neutral,positive
rating,,,
1,1562,0,0
2,249,0,0
3,0,279,0
4,0,0,438
5,0,0,3312


## 8. Sentiment classification

The documented experiment in `src/review_analytics/sentiment.py` uses `clean_text_new` as its only feature and the existing `sentiment` column as its target. App name, rating, and other metadata are excluded. A stratified 80/20 split with `random_state=42` produced 4,672 training and 1,168 test records. The vectorizer was fitted on training text only.

- **Majority baseline:** predicts the most frequent training label (`positive`) for every test record.
- **Text model:** TF-IDF unigram and bigram features (`min_df=1`, `max_features=50,000`, `sublinear_tf=True`, `lowercase=False`) followed by multinomial Logistic Regression (`max_iter=1,000`, `random_state=42`, default class weighting).
- **Evaluation:** accuracy, macro precision, macro recall, macro F1, and per-class metrics/support on the single held-out split.

### Previously documented held-out results

| Metric | Majority baseline | TF-IDF + Logistic Regression |
|---|---:|---:|
| Accuracy | 0.642123 | 0.879281 |
| Macro precision | 0.214041 | 0.578136 |
| Macro recall | 0.333333 | 0.604594 |
| Macro F1 | 0.260688 | 0.591040 |

| Logistic Regression class | Precision | Recall | F1 | Test support |
|---|---:|---:|---:|---:|
| Negative | 0.833780 | 0.859116 | 0.846259 | 362 |
| Neutral | 0.000000 | 0.000000 | 0.000000 | 56 |
| Positive | 0.900629 | 0.954667 | 0.926861 | 750 |

**Critical limitation:** the classifier predicted zero neutral test records. The apparent overall and macro improvements over the baseline do not make it a reliable three-class classifier. Also, the metrics measure agreement with existing labels of unverified provenance, not independently validated customer sentiment. This notebook reports the saved findings and does not train or evaluate a model.


## 9. Keyword-based customer-issue analysis

The existing detector in `src/review_analytics/issues.py` searches `review_text` for configured whole-word or phrase terms in six themes. Each review is counted at most once within a theme, but may match multiple themes, so counts overlap. Terms are not sentiment-aware.

**These are candidate keyword matches, not confirmed complaints.** Terms can appear in positive, neutral, ambiguous, or non-complaint contexts; unlisted wording and paraphrases can be missed. The reported values below are documented results from `ISSUE_ANALYSIS_REPORT.md`, not newly computed by this notebook.

| Candidate theme | Matching reviews | Denominator | Candidate-match rate |
|---|---:|---:|---:|
| Delivery and fulfilment | 638 | 5,840 | 10.92% |
| Refunds and cancellations | 335 | 5,840 | 5.74% |
| Customer support | 291 | 5,840 | 4.98% |
| App functionality | 147 | 5,840 | 2.52% |
| Payment issues | 115 | 5,840 | 1.97% |
| Product quality | 52 | 5,840 | 0.89% |

The report describes a 13-record sample with annotations, but also records a discrepancy between that sample and the pipeline's pending-validation status; annotation provenance is not documented. Aggregate themes remain marked not validated. The sample therefore does not establish the accuracy or prevalence of complaints across the dataset. No review text or sample records are embedded here.


## 10. Business implications and recommendations

The observed app-level rating differences and candidate keyword matches can guide questions for further investigation, but cannot explain why experiences differ or establish that one app is better for its full customer base.

1. Prioritize a documented human review of a larger, representative sample, with particular attention to fulfilment, refunds/cancellations, and support as candidate areas. Establish annotation guidance, record reviewer provenance, and assess agreement before treating themes as validated.
2. Confirm the sentiment-label source and assignment method. If independent sentiment evaluation is required, use separately and transparently annotated review text.
3. Do not use the current classifier for operational decisions as a three-class sentiment system: it predicted no neutral test records and its labels have unverified provenance.
4. Before drawing conclusions beyond these records, establish data provenance and coverage and test any proposed business intervention with suitable operational data and an appropriate evaluation design.

These recommendations follow the existing final report; they are validation steps, not claims that an operational problem or causal effect has been established.


## 11. Association between app and existing sentiment labels

This section tests whether `app_name` and the dataset's existing `sentiment` labels are statistically associated using a contingency table and Pearson chi-square test of independence. It reports the test statistic, degrees of freedom, p-value, expected counts, and Cramér’s V. The expected-count checks flag whether any expected cell count is below 1 and whether at least 80% are 5 or greater. The saved output displays `p-value: 0` because the very small value underflows or rounds to zero in the display; report it as `p < .001`, not as an exact zero.

The test assumes independent observations and sufficiently large expected cell counts; independence cannot be verified from this table alone. Regardless of the p-value, interpretation is limited: sentiment-label provenance is unverified, and the project audit found the labels align exactly with the tested rating mapping. The result concerns association with these existing rating-aligned labels, not independently validated customer sentiment, causation, or app quality. The significance threshold is alpha = 0.05.


In [3]:
import math

from scipy.stats import chi2_contingency

analysis_data = reviews[['app_name', 'sentiment']]
if analysis_data.isna().any().any():
    raise ValueError('Missing app_name or sentiment values are present; no records were dropped.')

blank_values = analysis_data.astype('string').apply(
    lambda column: column.str.strip().eq('').fillna(False)
)
if blank_values.any().any():
    raise ValueError('Blank app_name or sentiment values are present; no records were dropped.')

analysis_data = analysis_data.astype(str)
contingency_table = pd.crosstab(analysis_data['app_name'], analysis_data['sentiment'])
if contingency_table.shape[0] < 2 or contingency_table.shape[1] < 2:
    raise ValueError('The test requires at least two app categories and two sentiment labels.')

chi2_result = chi2_contingency(contingency_table, correction=False)
chi2_statistic = chi2_result.statistic
degrees_of_freedom = chi2_result.dof
p_value = chi2_result.pvalue
expected_counts = pd.DataFrame(
    chi2_result.expected_freq,
    index=contingency_table.index,
    columns=contingency_table.columns,
)
sample_size = int(contingency_table.to_numpy().sum())
minimum_dimension = min(contingency_table.shape[0] - 1, contingency_table.shape[1] - 1)
cramers_v = math.sqrt(chi2_statistic / (sample_size * minimum_dimension))

expected_values = expected_counts.to_numpy()
cells_below_one = int((expected_values < 1).sum())
cells_below_five = int((expected_values < 5).sum())
expected_cell_count = expected_values.size
percent_at_least_five = 100 * (expected_values >= 5).sum() / expected_cell_count
assumptions_satisfied = cells_below_one == 0 and percent_at_least_five >= 80
alpha = 0.05

print('Observed contingency table (counts):')
display(contingency_table)
print('Expected counts under independence:')
display(expected_counts.round(2))
print(f'Chi-square statistic: {chi2_statistic:.6f}')
print(f'Degrees of freedom: {degrees_of_freedom}')
print(f'p-value: {p_value:.6g}')
print(f"Cramer's V: {cramers_v:.6f}")
print(f'Expected cells below 1: {cells_below_one} of {expected_cell_count}')
print(f'Expected cells below 5: {cells_below_five} of {expected_cell_count}')
print(f'Expected cells at least 5: {percent_at_least_five:.1f}% (80% guideline)')
print(f'Expected-count guideline met: {assumptions_satisfied}')
print('Independence of observations: assumed; not verified by this table.')
if p_value < alpha:
    print(f'At alpha={alpha:.2f}, reject independence: the existing labels are associated with app_name.')
else:
    print(f'At alpha={alpha:.2f}, fail to reject independence: evidence of association is insufficient.')
print('Interpretation is limited to existing rating-aligned labels with unverified provenance; this is not independently validated sentiment.')


Observed contingency table (counts):


sentiment,negative,neutral,positive
app_name,,,
Amazon Shopping,1208,135,621
Flipkart,325,116,1459
Myntra,278,28,1670


Expected counts under independence:


sentiment,negative,neutral,positive
app_name,,,
Amazon Shopping,609.04,93.83,1261.13
Flipkart,589.20,90.77,1220.03
Myntra,612.76,94.40,1268.84


Chi-square statistic: 1460.739675
Degrees of freedom: 4
p-value: 0
Cramer's V: 0.353643
Expected cells below 1: 0 of 9
Expected cells below 5: 0 of 9
Expected cells at least 5: 100.0% (80% guideline)
Expected-count guideline met: True
Independence of observations: assumed; not verified by this table.
At alpha=0.05, reject independence: the existing labels are associated with app_name.
Interpretation is limited to existing rating-aligned labels with unverified provenance; this is not independently validated sentiment.


## 12. Limitations, reproducibility, and conclusion

### Limitations

- **Scope and provenance:** the available data cover three apps, English-language records from India, and one recorded collection timestamp. Upstream acquisition and preprocessing history and representativeness beyond these files are not fully established.
- **Labels:** the sentiment-label provenance is unverified and labels align exactly with the tested rating mapping. Classification metrics are agreement scores against these labels, not independent sentiment validation.
- **Classification:** results come from one stratified random holdout. The model predicted zero neutral test records; stability across seeds, users, time, or app-held-out data was not tested.
- **Issue themes:** keyword matches are not contextual or validated complaint classifications. The limited sample has a documented validation-status discrepancy and unknown annotation provenance.
- **Descriptive inference:** observed app-level differences do not demonstrate causes or generalize to all app users. The 500-character maximum does not by itself prove truncation.

### Reproducibility

Run the lightweight notebook cells from the project root with the local file `data/processed/google_play_reviews_cleaned.csv` available. The environment check previously recorded Python 3.14.7 and Pandas 3.0.6. The summary cells read data and display basic tables only; they do not write files. The EDA, sentiment-model, and issue-analysis pipelines are intentionally not run here. Their source and existing reports document the methods and prior results. The project does not pin dependency versions, and its local data and generated outputs are excluded from the public repository.

Relevant project evidence: `DATA_VALIDATION_REPORT.md`, `CLEANING_REPORT.md`, `EDA_REPORT.md`, `SENTIMENT_MODEL_REPORT.md`, `ISSUE_ANALYSIS_REPORT.md`, and `FINAL_PROJECT_REPORT.md`.

### Conclusion

The existing reports document the main dataset distributions, a TF-IDF/Logistic Regression model with higher overall and macro scores than the majority baseline, and keyword matches concentrated most often in delivery and fulfilment. Interpretation must remain cautious: sentiment labels have unverified provenance, the model did not predict any neutral test records, and keyword matches are candidate themes rather than validated complaints. These findings support further data and human validation, not claims of independently measured sentiment, confirmed complaint prevalence, causation, or app-wide customer experience.
